# Byte-Pair Encoding tokenization (BPE)

- Byte-Pair Encoding (BPE) was initially developed as an algorithm to compress texts, and then used by OpenAI for tokenization when pretraining the GPT model.
- It’s used by a lot of Transformer models, including GPT, GPT-2, RoBERTa, BART, and DeBERTa.

## Training algorithm

### Create base vocabulary

- BPE training starts by computing the unique set of words used in the corpus (after the normalization and pre-tokenization steps are completed)
- then building the vocabulary by taking all the symbols used to write those words.
- As a very simple example, let’s say our corpus uses these five words: ```"hug", "pug", "pun", "bun", "hugs"```
    - The base vocabulary will then be `["b", "g", "h", "n", "p", "s", "u"]`.
- For real-world cases, that base vocabulary will contain all the **ASCII** characters, at the very least, and probably some **Unicode** characters as well.
- If an example you are tokenizing uses a character that is not in the training corpus, that character will be converted to the **unknown token**.
    - That’s one reason why lots of NLP models are very bad at analyzing content with emojis, for instance.
- The **GPT-2 and RoBERTa** tokenizers (which are pretty similar) have a clever way to deal unknown tokens:
    - they don’t look at words as being written with Unicode characters, but with **bytes**.
    - This way the base vocabulary has a small size (256),
    - but every character you can think of will still be included and not end up being converted to the unknown token.
    - This trick is called **byte-level BPE**.

### Merge

- After getting this base vocabulary, we add new tokens until the desired **vocabulary size** is reached by learning **merges**, which are **rules** to merge two elements of the existing vocabulary together into a new one.
- So, at the beginning these merges will create tokens with two characters, and then, as training progresses, longer subwords.
- At any step during the tokenizer training, the BPE algorithm will search for the **most frequent pair** of existing tokens (by “pair,” here we mean **two consecutive tokens** in a word).
- That most frequent pair is the one that will be merged, and we rinse and repeat for the next step.
- Going back to our previous example, let’s assume the words had the following frequencies: ```("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)```

current vocab --> ```["b", "g", "h", "n", "p", "s", "u"]```

current corpus --> ```("h" "u" "g", 10), ("p" "u" "g", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "u" "g" "s", 5)```

---
most frequent pair -->  ```("u", "g")``` for 20 times

new rule --> ```("u", "g") -> "ug"```

---
updated vocab --> ```["b", "g", "h", "n", "p", "s", "u", "ug"]```

updated corpus --> ```("h" "ug", 10), ("p" "ug", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "ug" "s", 5)```

---
most frequent pair -->  ```("u", "n")``` for 16 times

new rule --> ```("u", "n") -> "un"```

---
updated vocab --> ```["b", "g", "h", "n", "p", "s", "u", "ug", "un"]```

updated corpus -->  ```("h" "ug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("h" "ug" "s", 5)```


---
most frequent pair -->  ```("h", "ug")``` for 15 times

new rule --> ```("h", "ug") -> "hug"```

---
updated vocab --> ```["b", "g", "h", "n", "p", "s", "u", "ug", "un", "hug"]```

updated corpus --> ```("hug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("hug" "s", 5)```

---
And we continue like this until we reach the desired vocabulary size.

## Tokenization algorithm

- Tokenization follows the training process closely, in the sense that new inputs are tokenized by applying the following steps:
    1.  Normalization
    2.  Pre-tokenization
    3.  Splitting the words into individual characters
    4.  Applying the merge rules learned in order on those splits

- Let’s take the example we used during training, with the three merge rules learned:
```
("u", "g") -> "ug"
("u", "n") -> "un"
("h", "ug") -> "hug"
```

---

- `"bug"` --> `["b", "ug"]`.
- `"mug"` --> `["[UNK]", "ug"]`
- `"thug"` --> `["[UNK]", "hug"]`
- `"unhug"` --> `["un", "hug"]`

## Implementation

- This won’t be an optimized version you can actually use on a big corpus
- We just want to see the code so we can understand the algorithm a little bit better.
- First, We need to **pre-tokenize** that corpus into words.
    - Since we are replicating a BPE tokenizer (like GPT-2), we will use the gpt2 tokenizer for the pre-tokenization
- The next step is to compute the base vocabulary, formed by all the characters used in the corpus.
- We also add the **special tokens** used by the model at the beginning of that vocabulary.
    - In the case of GPT-2, the only special token is `"<|endoftext|>"`
- Next, we split each word into individual characters, to be able to start training.

In [ ]:
from transformers import AutoTokenizer
from collections import defaultdict

corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

tokenizer = AutoTokenizer.from_pretrained("gpt2")
word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)
print("==========================")

alphabet = []
for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()
vocab = ["<|endoftext|>"] + alphabet.copy()
print(alphabet)
print("==========================")
print(vocab)
print("==========================")
splits = {word: [c for c in word] for word in word_freqs.keys()}
for k,v in splits.items(): print(f"{k}: {v}")
print("==========================")

- Now that we are ready for training, let’s write a function that computes the frequency of each pair.
- We’ll need to use this at each step of the training.

In [ ]:
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

pair_freqs = compute_pair_freqs(splits)

for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break
print("=====================")

best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)
print("=====================")

- So the first merge to learn is `('Ġ', 't') -> 'Ġt'`, and we add `'Ġt'` to the vocabulary
- To continue, we need to apply that merge in our `splits` dictionary.
    - Let’s write another function for this

In [ ]:
merges = {("Ġ", "t"): "Ġt"}
vocab.append("Ġt")

def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

splits = merge_pair("Ġ", "t", splits)
print(splits["Ġtrained"])

- Now we have everything we need to loop until we have learned all the merges we want. Let’s aim for a vocab size of 50
- 💡 Using `train_new_from_iterator()` on the same corpus won’t result in the exact same vocabulary.
    - This is because when there is a choice of the most frequent pair, we selected the first one encountered
    - while the 🤗 Tokenizers library selects the first one based on its inner IDs.

In [ ]:
vocab_size = 50
while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

print(merges)
print("=====================")
print(vocab)
print("=====================")

## Inference

In [ ]:
def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    splits = [[l for l in word] for word in pre_tokenized_text]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])

tokenize("This is not a token.")

⚠️ Our implementation will throw an error if there is an unknown character since we didn’t do anything to handle them.
- GPT-2 doesn’t actually have an unknown token (it’s impossible to get an unknown character when using byte-level BPE)
- but this could happen here because we did not include all the possible bytes in the initial vocabulary.

# WordPiece tokenization

- WordPiece is the tokenization algorithm Google developed to pretrain BERT.
- It has since been reused in quite a few Transformer models based on BERT, such as DistilBERT, MobileBERT, Funnel Transformers, and MPNET.
- It’s very similar to BPE in terms of the training, but the actual tokenization is done differently.

## Training algorithm

- ⚠️ Google never open-sourced its implementation of the training algorithm of WordPiece
- So, what follows is our best guess based on the published literature. It may not be 100% accurate.
- Like BPE, WordPiece starts from a small vocabulary including the special tokens used by the model and the initial alphabet.
- It identifies subwords by adding a prefix (like `##` for BERT), each word is initially split by adding that prefix to all the characters inside the word.
- Then, again like BPE, WordPiece learns merge rules.
- **The main difference** is the way the pair to be merged is selected. Instead of selecting the most frequent pair, WordPiece computes a **score** for each pair, using the following formula

$score=(freq\_of\_pair)/(freq\_of\_first\_element×freq\_of\_second\_element)$

- By dividing the frequency of the pair by the product of the frequencies of each of its parts, the algorithm prioritizes the merging of pairs where the individual parts are less frequent in the vocabulary.
    - For instance, it won’t necessarily merge `("un", "##able")` even if that pair occurs very frequently in the vocabulary, because the two pairs `"un"` and `"##able"` will likely each appear in a lot of other words and have a high frequency.
    - In contrast, a pair like `("hu", "##gging")` will probably be merged faster (assuming the word “hugging” appears often in the vocabulary) since `"hu"` and `"##gging"` are likely to be less frequent individually.


---
Let’s look at the same vocabulary we used in the BPE training example (forget about special tokens for now!!!)

initial vocab --> ```["b", "h", "p", "##g", "##n", "##s", "##u"]```

initial corpus --> ```("h" "##u" "##g", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("h" "##u" "##g" "##s", 5)```

---
best pair --> `("##g", "##s")` with a score of 1 / 20

new merge rule --> `("##g", "##s") -> ("##gs")`.

---
updated vocab --> ```["b", "h", "p", "##g", "##n", "##s", "##u", "##gs"]```

updated corpus --> ```("h" "##u" "##g", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("h" "##u" "##gs", 5)```

---
best pair --> `("h", "##u")` with a score of 1 / 38 (equal to all other pairs since all of them include `"##u"`)

new merge rule --> `("h", "##u") -> "hu"`.

---
updated vocab --> ```["b", "h", "p", "##g", "##n", "##s", "##u", "##gs", "hu"]```

updated corpus --> ```("hu" "##g", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("hu" "##gs", 5)```

---
best pair --> `("hu", "##g")` and `("hu", "##gs")` with 1/15

new merge rule --> `("hu", "##g") -> "hug"`

---
updated vocab --> ```["b", "h", "p", "##g", "##n", "##s", "##u", "##gs", "hu", "hug"]```

updated corpus --> ```("hug", 10), ("p" "##u" "##g", 5), ("p" "##u" "##n", 12), ("b" "##u" "##n", 4), ("hu" "##gs", 5)```


---
and we continue like this until we reach the desired vocabulary size.

## Tokenization algorithm

- Tokenization **differs** in WordPiece and BPE in that WordPiece only saves the final vocabulary, **not the merge rules** learned.
- Starting from the word to tokenize, WordPiece finds the **longest subword** that is in the vocabulary, then splits on it.
- For instance, if we use the vocabulary learned in the example above, for the word `"hugs"`
    - the longest subword starting from the beginning that is inside the vocabulary is `"hug"`, so we split there and get `["hug", "##s"]`.
    - We then continue with `"##s"`, which is in the vocabulary, so the tokenization of `"hugs"` is `["hug", "##s"]`.
    - With BPE, we would have applied the merges learned in order and tokenized this as `["hu", "##gs"]`, so the encoding is different.
- As another example, let’s see how the word `"bugs"` would be tokenized.
    - `"b"` is the longest subword starting at the beginning of the word that is in the vocabulary, so we split there and get `["b", "##ugs"]`.
    - Then `"##u"` is the longest subword starting at the beginning of `"##ugs"` that is in the vocabulary, so we split there and get `["b", "##u, "##gs"]`.
    - Finally, `"##gs"` is in the vocabulary, so this last list is the tokenization of `"bugs"`.
    - With BPE it will be `"bugs"` --> `["b", "ug", "s]`.
- When the tokenization gets to a stage where it’s not possible to find a subword in the vocabulary, the **whole word is tokenized as unknown**
    - So, for instance, `"bum"` will just be `["[UNK]"]`, not `["b", "##u", "[UNK]"]`. 
    - This is **another difference from BPE**, which would only classify the individual characters not in the vocabulary as unknown.

## Implementation

In [ ]:
from transformers import AutoTokenizer
from collections import defaultdict

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)
print("=====================")

alphabet = []
for word in word_freqs.keys():
    if word[0] not in alphabet:
        alphabet.append(word[0])
    for letter in word[1:]:
        if f"##{letter}" not in alphabet:
            alphabet.append(f"##{letter}")

alphabet.sort()

print(alphabet)
print("=====================")


vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + alphabet.copy()
print(vocab)
print("=====================")


splits = {
    word: [c if i == 0 else f"##{c}" for i, c in enumerate(word)]
    for word in word_freqs.keys()
}
for k,v in splits.items(): print(f"{k}: {v}")
print("==========================")

- Now that we are ready for training, let’s write a function that computes the frequency of each pair.
- We’ll need to use this at each step of the training.

In [ ]:
def compute_pair_scores(splits):
    letter_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            letter_freqs[split[i]] += freq
            pair_freqs[pair] += freq
        letter_freqs[split[-1]] += freq

    scores = {
        pair: freq / (letter_freqs[pair[0]] * letter_freqs[pair[1]])
        for pair, freq in pair_freqs.items()
    }
    return scores

pair_scores = compute_pair_scores(splits)
for i, key in enumerate(pair_scores.keys()):
    print(f"{key}: {pair_scores[key]}")
    if i >= 5:
        break
print("====================")


best_pair = ""
max_score = None
for pair, score in pair_scores.items():
    if max_score is None or max_score < score:
        best_pair = pair
        max_score = score

print(best_pair, max_score)
print("====================")

- So the first merge to learn is `('a', '##b') -> 'ab'`, and we add `'ab'` to the vocabulary
- To continue, we need to apply that merge in our splits dictionary. Let’s write another function for this

In [ ]:
vocab.append("ab")
print(f"last five elemenets of the updated vocab: {vocab[-5:]}")
print("==================")

def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue
        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                merge = a + b[2:] if b.startswith("##") else a + b
                split = split[:i] + [merge] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

splits = merge_pair("a", "##b", splits)
print(splits["about"])
print("==================")

- Now we have everything we need to loop until we have learned all the merges we want. Let’s aim for a vocab size of 70
- 💡 Using `train_new_from_iterator()` on the same corpus won’t result in the exact same vocabulary.
    - This is because the 🤗 Tokenizers library does not implement WordPiece for the training (since we are not completely sure of its internals), but uses BPE instead.

In [ ]:
vocab_size = 70
while len(vocab) < vocab_size:
    scores = compute_pair_scores(splits)
    best_pair, max_score = "", None
    for pair, score in scores.items():
        if max_score is None or max_score < score:
            best_pair = pair
            max_score = score
    splits = merge_pair(*best_pair, splits)
    new_token = (
        best_pair[0] + best_pair[1][2:]
        if best_pair[1].startswith("##")
        else best_pair[0] + best_pair[1]
    )
    vocab.append(new_token)

print(vocab)

## Inference

In [ ]:
def encode_word(word):
    tokens = []
    while len(word) > 0:
        i = len(word)
        while i > 0 and word[:i] not in vocab:
            i -= 1
        if i == 0:
            return ["[UNK]"]
        tokens.append(word[:i])
        word = word[i:]
        if len(word) > 0:
            word = f"##{word}"
    return tokens

print(encode_word("Hugging"))
print(encode_word("HOgging"))
print("======================")


def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    encoded_words = [encode_word(word) for word in pre_tokenized_text]
    return sum(encoded_words, [])

print(tokenize("This is the Hugging Face course!"))
print("======================")

# Unigram tokenization

- The Unigram algorithm is used in combination with [SentencePiece](https://huggingface.co/papers/1808.06226)
- It is used by models like AlBERT, T5, mBART, Big Bird, and XLNet.
- SentencePiece addresses the fact that not all languages use spaces to separate words.
- Instead, SentencePiece treats the input as a raw input stream which includes the space in the set of characters to use.
- Then it can use the Unigram algorithm to construct the appropriate vocabulary.

## Training algorithm

- Compared to BPE and WordPiece, Unigram works in **the other direction**:
    - it starts from a big vocabulary and removes tokens from it until it reaches the desired vocabulary size.
    - There are several options to use to build that base vocabulary
        - we can take the most common substrings in pre-tokenized words
        - or apply BPE on the initial corpus with a large vocabulary size.

- At each step of the training, the Unigram algorithm computes a **loss** over the corpus given the current vocabulary.
- Then, for each symbol in the vocabulary, the algorithm computes how much the overall loss would increase if the symbol was removed
    - and looks for the symbols that would increase it the least.
    - Those symbols have a lower effect on the overall loss over the corpus,
    - so in a sense they are “less needed” and are the best candidates for removal.
    - This is all a very **costly operation**, so we don’t just remove the single symbol associated with the lowest loss increase
        - but the `p` (usually 10 or 20) percent of the symbols associated with the lowest loss increase.
- This process is then repeated until the vocabulary has reached the desired size.
- Note that we **never** remove the **base characters**, to make sure any word can be tokenized.

Let's consider
- initial corpus --> ```("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)```
- initial vocab --> ```["h", "u", "g", "hu", "ug", "p", "pu", "n", "un", "b", "bu", "s", "hug", "gs", "ugs"]```

## Tokenization algorithm

- A Unigram model is a type of language model that considers each token to be **independent** of the tokens **before** it.
- It’s the simplest language model, in the sense that the probability of token X given the previous context is just the probability of token X.
    - So, if we used a Unigram language model to generate text, we would always predict the most common token.

- The probability of a given token is its frequency (the number of times we find it) in the original corpus, divided by the sum of all frequencies of all tokens in the vocabulary (to make sure the probabilities sum up to 1).
- Here are the frequencies of all the possible subwords in the vocabulary:
```
("h", 15) ("u", 36) ("g", 20) ("hu", 15) ("ug", 20) ("p", 17) ("pu", 17) ("n", 16)
("un", 16) ("b", 4) ("bu", 4) ("s", 5) ("hug", 15) ("gs", 5) ("ugs", 5)
```

- Now, **to tokenize** a given word, we look at all the possible segmentations into tokens and compute the probability of each according to the Unigram model.
- Since all tokens are considered **independent**, this probability is just the **product** of the probability of each token.
- In the example of `"pug"`, here are the probabilities we would get for each possible segmentation:
```
["p", "u", "g"] : 0.000389
["p", "ug"] : 0.0022676
["pu", "g"] : 0.0022676
```
- In this case, it was easy to find all the possible segmentations and compute their probabilities, but in general it’s going to be a bit harder. There is a classic algorithm used for this, called the **Viterbi algorithm**.

### Viterbi algorithm
- Essentially, we can build a graph to detect the possible segmentations of a given word by saying there is a branch from character _a_ to character _b_ if the subword from _a_ to _b_ is in the vocabulary, and attribute to that branch the probability of the subword.
- To find the **path** in that graph that is going to have the **best score** the Viterbi algorithm determines,
    - for **each position** in the word, the **segmentation** with the best score that ends at that position.
- Since we go from the beginning to the end, that best score can be found by looping through all subwords ending at the current position and then using the best tokenization score from the position this subword begins at.
- Then, we just have to unroll the path taken to arrive at the end.

- Let’s take a look at an example using our vocabulary and the word `"unhug"`.
    - For each position, the subwords with the best scores ending there are the following:
```
Character 0 (u): "u" (score 0.171429)
Character 1 (n): "un" (score 0.076191)
Character 2 (h): "un" "h" (score 0.005442)
Character 3 (u): "un" "hu" (score 0.005442)
Character 4 (g): "un" "hug" (score 0.005442)
```

## Back to training

- At any given stage, this loss is computed by tokenizing every word in the corpus, using the current vocabulary and the Unigram model determined by the frequencies of each token in the corpus (as seen before).
- Each word in the corpus has a score, and the loss is the **negative log likelihood** of those scores
    - that is, the sum for all the words in the corpus of all the `-log(P(word))`.

```
"hug": ["hug"] (score 0.071428)
"pug": ["pu", "g"] (score 0.007710)
"pun": ["pu", "n"] (score 0.006168)
"bun": ["bu", "n"] (score 0.001451)
"hugs": ["hug", "s"] (score 0.001701)
```
- the loss is
```
10 * (-log(0.071428)) + 5 * (-log(0.007710)) + 12 * (-log(0.006168)) + 4 * (-log(0.001451)) + 5 * (-log(0.001701)) = 169.8
```

- Now we need to compute how removing each token affects the loss.
- For example, removing `"hug"` will make the loss worse, because the tokenization of `"hug"` and `"hugs"` will become:
```
"hug": ["hu", "g"] (score 0.006802)
"hugs": ["hu", "gs"] (score 0.001701)
```
- These changes will cause the loss to rise by:

```- 10 * (-log(0.071428)) + 10 * (-log(0.006802)) = 23.5```

## Implementation

- Like for BPE and WordPiece, we begin by counting the number of occurrences of each word in the corpus
- Then, we need to initialize our vocabulary to something larger than the vocab size we will want at the end.
    - We have to include all the basic characters (otherwise we won’t be able to tokenize every word)
    - But for the bigger substrings we’ll only keep the most common ones, so we sort them by frequency
    - We group the characters with the best subwords to arrive at an initial vocabulary of size 300
    - **SentencePiece** uses a more efficient algorithm called **Enhanced Suffix Array (ESA)** to create the initial vocabulary.

In [ ]:
from transformers import AutoTokenizer
from collections import defaultdict

tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")

corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)
print("==================")


char_freqs = defaultdict(int)
subwords_freqs = defaultdict(int)
for word, freq in word_freqs.items():
    for i in range(len(word)):
        char_freqs[word[i]] += freq
        # Loop through the subwords of length at least 2
        for j in range(i + 2, len(word) + 1):
            subwords_freqs[word[i:j]] += freq

# Sort subwords by frequency
sorted_subwords = sorted(subwords_freqs.items(), key=lambda x: x[1], reverse=True)
print(sorted_subwords[:10])
print("==================")


token_freqs = list(char_freqs.items()) + sorted_subwords[: 300 - len(char_freqs)]
token_freqs = {token: freq for token, freq in token_freqs}
print(token_freqs)
print("==================")

- Next, we compute the sum of all frequencies, to convert the frequencies into probabilities.
    - For our model we will store the logarithms of the probabilities, because it’s more numerically stable to add logarithms than to multiply small numbers
    - and this will simplify the computation of the loss of the model

In [ ]:
from math import log

total_sum = sum([freq for token, freq in token_freqs.items()])
model = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}

### Viterbi algo

- We will store one dictionary per position in the word (from 0 to its total length), with two keys
    - the index of the start of the last token in the best segmentation
    - and the score of the best segmentation.
- With the index of the start of the last token, we will be able to retrieve the full segmentation once the list is completely populated.

- Populating the list is done with just two loops:
    - the main loop goes over each start position
    - and the second loop tries all substrings beginning at that start position.
        - If the substring is in the vocabulary, we have a new segmentation of the word up until that end position, which we compare to what is in `best_segmentations`.

- Once the main loop is finished, we just start from the end and hop from one start position to the next, recording the tokens as we go, until we reach the start of the word

In [ ]:
def encode_word(word, model):
    best_segmentations = [{"start": 0, "score": 1}] + [
        {"start": None, "score": None} for _ in range(len(word))
    ]
    for start_idx in range(len(word)):
        # This should be properly filled by the previous steps of the loop
        best_score_at_start = best_segmentations[start_idx]["score"]
        for end_idx in range(start_idx + 1, len(word) + 1):
            token = word[start_idx:end_idx]
            if token in model and best_score_at_start is not None:
                score = model[token] + best_score_at_start
                # If we have found a better segmentation ending at end_idx, we update
                if (
                    best_segmentations[end_idx]["score"] is None
                    or best_segmentations[end_idx]["score"] > score
                ):
                    best_segmentations[end_idx] = {"start": start_idx, "score": score}

    segmentation = best_segmentations[-1]
    if segmentation["score"] is None:
        # We did not find a tokenization of the word -> unknown
        return ["<unk>"], None

    score = segmentation["score"]
    start = segmentation["start"]
    end = len(word)
    tokens = []
    while start != 0:
        tokens.insert(0, word[start:end])
        next_start = best_segmentations[start]["start"]
        end = start
        start = next_start
    tokens.insert(0, word[start:end])
    return tokens, score

print(encode_word("Hopefully", model))
print(encode_word("This", model))

### Compute loss

In [ ]:
def compute_loss(model):
    loss = 0
    for word, freq in word_freqs.items():
        _, word_loss = encode_word(word, model)
        loss += freq * word_loss
    return loss

print(compute_loss(model))

In [ ]:
import copy
def compute_scores(model):
    scores = {}
    model_loss = compute_loss(model)
    for token, score in model.items():
        # We always keep tokens of length 1
        if len(token) == 1:
            continue
        model_without_token = copy.deepcopy(model)
        _ = model_without_token.pop(token)
        scores[token] = compute_loss(model_without_token) - model_loss
    return scores

scores = compute_scores(model)
print(scores["ll"])
print(scores["his"])

### Train loop

In [ ]:
percent_to_remove = 0.1
while len(model) > 100:
    scores = compute_scores(model)
    sorted_scores = sorted(scores.items(), key=lambda x: x[1])
    # Remove percent_to_remove tokens with the lowest scores.
    for i in range(int(len(model) * percent_to_remove)):
        _ = token_freqs.pop(sorted_scores[i][0])

    total_sum = sum([freq for token, freq in token_freqs.items()])
    model = {token: -log(freq / total_sum) for token, freq in token_freqs.items()}

## Inference

- The XLNetTokenizer uses SentencePiece which is why the `"_"` character is included.
- To decode with SentencePiece, concatenate all the tokens and replace `"_"` with a space.

In [ ]:
def tokenize(text, model):
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in words_with_offsets]
    encoded_words = [encode_word(word, model)[0] for word in pre_tokenized_text]
    return sum(encoded_words, [])


print(tokenize("This is the Hugging Face course.", model))